# Qwen2.5-1.5B Fine-Tuning on ROCm (AMD GPU)

**Task:** Supervised fine-tuning of `Qwen/Qwen2.5-1.5B-Instruct` on a JSONL chat dataset.  
**Backend:** ROCm / HIP (AMD GPU) — no CUDA, no bitsandbytes.  
**PEFT method:** LoRA via `peft`.  
**Evaluation metrics:** ROUGE-1/2/L, BLEU, SacreBLEU (before & after training).

---
**Notebook Structure**
1. Environment & dependency checks  
2. Config & constants  
3. Directory setup  
4. Model & tokenizer loading  
5. Data loading & sampling  
6. Dataset formatting & tokenization  
7. Baseline evaluation (pre-training)  
8. LoRA setup & training  
9. Post-training evaluation  
10. Save adapter & results  


## 1. Environment & Dependency Check

In [ ]:
# ── Install required packages (run once) ─────────────────────────────────────
# Uncomment the block below if running for the first time.

# !pip install -q transformers==4.44.0 peft==0.12.0 accelerate==0.34.0 \
#              datasets==2.21.0 trl==0.10.1 \
#              rouge-score sacrebleu nltk


In [ ]:
import torch

# ── ROCm / HIP device check ───────────────────────────────────────────────────
# On ROCm, torch.cuda.* APIs map to AMD GPUs via HIP.
# If this shows False, ensure ROCm-compatible PyTorch is installed.
print(f"PyTorch version : {torch.__version__}")
print(f"ROCm/HIP available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device      : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will run on CPU (very slow).")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device    : {DEVICE}")


## 2. Config & Constants

All tunable parameters live here — easy to adjust without touching the rest of the notebook.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR        = "/workspace/shared"
MODEL_SAVE_DIR  = f"{BASE_DIR}/models/qwen2.5-1.5b"
ADAPTER_SAVE_DIR= f"{BASE_DIR}/models/qwen2.5-1.5b-lora"
CHECKPOINT_DIR  = f"{BASE_DIR}/checkpoints"
RESULTS_DIR     = f"{BASE_DIR}/results"

# Dataset paths (JSONL format; each line is a JSON object with a "messages" key)
TRAIN_PATH = f"{BASE_DIR}/CFPB-Dataset-for-qwen/train.jsonl"
VAL_PATH   = f"{BASE_DIR}/CFPB-Dataset-for-qwen/validation.jsonl"
TEST_PATH  = f"{BASE_DIR}/CFPB-Dataset-for-qwen/test.jsonl"

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # HuggingFace model id

# ── Dataset sizes ─────────────────────────────────────────────────────────────
N_TRAIN = 500   # samples used for training
N_VAL   = 250   # samples used for validation
N_TEST  = 250   # samples used for evaluation

# Seed for reproducible random sampling
RANDOM_SEED = 42

# ── Tokenization ──────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 1024   # max token length per example

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R        = 16      # LoRA rank (adapter bottleneck dimension)
LORA_ALPHA    = 32      # LoRA scaling factor
LORA_DROPOUT  = 0.05    # dropout in LoRA layers
# Attention projection layers to apply LoRA on
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Training hyperparameters ──────────────────────────────────────────────────
NUM_EPOCHS              = 3
BATCH_SIZE              = 2     # per-device train batch size
GRADIENT_ACCUM_STEPS    = 8     # effective batch = BATCH_SIZE * GRADIENT_ACCUM_STEPS
LEARNING_RATE           = 2e-4
LOGGING_STEPS           = 10
EVAL_STEPS              = 25
SAVE_STEPS              = 25

# ── Inference (evaluation) ────────────────────────────────────────────────────
MAX_NEW_TOKENS = 128    # max tokens to generate per prediction


## 3. Directory Setup

In [ ]:
import os

def setup_directories(*dirs: str) -> None:
    """Create required output directories if they don't already exist."""
    for d in dirs:
        os.makedirs(d, exist_ok=True)
        print(f"  Ready: {d}")

print("Setting up directories...")
setup_directories(MODEL_SAVE_DIR, ADAPTER_SAVE_DIR, CHECKPOINT_DIR, RESULTS_DIR)


## 4. Model & Tokenizer Loading

We load in **bfloat16** — the natural precision for ROCm GPUs (AMD MI-series cards support bf16 natively).  
`bitsandbytes` is CUDA-only and must **not** be used here; we skip 4-bit quantization entirely.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model_and_tokenizer(model_name: str, device: str):
    """
    Load Qwen model and tokenizer.

    ROCm note: We use bfloat16 instead of float16 for better numerical
    stability on AMD GPUs. bitsandbytes 4-bit quantization is CUDA-only
    and is intentionally omitted here.

    Returns
    -------
    model     : AutoModelForCausalLM
    tokenizer : AutoTokenizer
    """
    print(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Qwen models may not have a padding token set by default
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model from : {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,   # bf16 is optimal on ROCm/AMD GPUs
        device_map="auto",            # lets HF pick the right GPU automatically
    )

    print(f"Model loaded — parameters: {model.num_parameters():,}")
    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME, DEVICE)


In [ ]:
# ── Save to local disk (optional — avoids re-downloading later) ──────────────
def save_model_and_tokenizer(model, tokenizer, save_path: str) -> None:
    """Persist model weights and tokenizer vocab to disk."""
    print(f"Saving to {save_path} ...")
    tokenizer.save_pretrained(save_path)
    model.save_pretrained(save_path)
    print("Saved.")

# Uncomment to save the base model locally:
# save_model_and_tokenizer(model, tokenizer, MODEL_SAVE_DIR)


## 5. Data Loading & Sampling

The dataset uses JSONL format — each line is a JSON object with a `"messages"` key containing a list of chat turns (OpenAI-style: `[{"role": ..., "content": ...}, ...]`).


In [ ]:
import json
import random
from typing import List, Dict, Any

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    """Read a JSONL file and return a list of parsed JSON objects."""
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:                        # skip blank lines
                data.append(json.loads(line))
    return data


def sample_data(data: List, n: int, seed: int = RANDOM_SEED) -> List:
    """
    Randomly sample up to `n` examples from `data`.
    Uses min(n, len(data)) so it never errors on small datasets.
    """
    random.seed(seed)
    return random.sample(data, min(n, len(data)))


# ── Load raw splits ───────────────────────────────────────────────────────────
print("Loading dataset splits...")
raw_train = load_jsonl(TRAIN_PATH)
raw_val   = load_jsonl(VAL_PATH)
raw_test  = load_jsonl(TEST_PATH)

print(f"  Full train size : {len(raw_train):,}")
print(f"  Full val size   : {len(raw_val):,}")
print(f"  Full test size  : {len(raw_test):,}")

# ── Sub-sample for this iteration ─────────────────────────────────────────────
train_data = sample_data(raw_train, N_TRAIN)
val_data   = sample_data(raw_val,   N_VAL)
test_data  = sample_data(raw_test,  N_TEST)

print(f"\nSampled splits:")
print(f"  Train : {len(train_data)}")
print(f"  Val   : {len(val_data)}")
print(f"  Test  : {len(test_data)}")

# Quick sanity-check — print one example
print("\nSample training example (messages):")
print(json.dumps(train_data[0]["messages"], indent=2)[:600], "...")


## 6. Dataset Formatting & Tokenization

We use the tokenizer's built-in **chat template** to convert conversation turns into a single training string, then tokenize with truncation.


In [ ]:
from datasets import Dataset

def format_chat(example: Dict[str, Any]) -> Dict[str, str]:
    """
    Apply the tokenizer's chat template to a single conversation example.

    The chat template converts the list of messages into a single string
    that includes special tokens (e.g., <|im_start|>, <|im_end|> for Qwen).
    `add_generation_prompt=False` because we include the full assistant turn.
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


def tokenize_example(example: Dict[str, Any]) -> Dict:
    """
    Tokenize the formatted text.

    truncation=True  : silently trims sequences longer than MAX_SEQ_LENGTH.
    padding=False    : padding is done per-batch by the DataCollator, not here.
    """
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,          # leave padding to DataCollatorForLanguageModeling
        max_length=MAX_SEQ_LENGTH,
    )


def build_hf_dataset(data: List[Dict]) -> Dataset:
    """
    Convert a list of raw examples into a tokenized HuggingFace Dataset.

    Steps
    -----
    1. Wrap list  -> HF Dataset
    2. format_chat: apply chat template (list-of-dicts -> single string)
    3. tokenize_example: string -> input_ids + attention_mask
    4. Drop all non-tensor columns.

    WHY the column drop is required
    --------------------------------
    The raw data has a 'messages' column (list of dicts) and 'text' (string).
    When the DataCollator tries to batch these into tensors it raises:

        ValueError: Unable to create tensor, you should probably activate
        truncation and/or padding ... features have excessive nesting.

    Keeping only 'input_ids' and 'attention_mask' avoids this entirely.
    """
    ds = Dataset.from_list(data)
    ds = ds.map(format_chat)
    ds = ds.map(tokenize_example, batched=True)

    # Keep only the columns the Trainer / DataCollator actually needs
    tensor_cols = {"input_ids", "attention_mask", "labels"}
    drop_cols   = [c for c in ds.column_names if c not in tensor_cols]
    ds = ds.remove_columns(drop_cols)
    return ds


print("Formatting and tokenizing datasets...")
train_ds = build_hf_dataset(train_data)
val_ds   = build_hf_dataset(val_data)
print(f"  Train dataset : {len(train_ds)} examples")
print(f"  Val dataset   : {len(val_ds)} examples")
print(f"  Columns kept  : {train_ds.column_names}")
print(f"\nSample tokenized entry (first 20 token ids):")
print(train_ds[0]["input_ids"][:20])


## 7. Evaluation Utilities

We use three complementary metrics:

| Metric | What it measures |
|--------|-----------------|
| **ROUGE-1/2/L** | n-gram recall between prediction and reference |
| **BLEU** | n-gram precision (corpus-level) |
| **SacreBLEU** | BLEU with standardised tokenisation (more reproducible) |

These are computed on the **assistant's last turn** (the model output we want to learn).


In [ ]:
from rouge_score import rouge_scorer
import sacrebleu
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def build_inference_prompt(messages: List[Dict]) -> str:
    """
    Build an inference-time prompt by stripping the final assistant turn.
    The model must *generate* that turn.
    """
    # All turns except the last assistant response
    prompt_messages = messages[:-1]
    return tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,   # add the opening of the assistant turn
    )


def predict(messages: List[Dict], model) -> str:
    """
    Run greedy decoding for a single conversation example.

    Returns the decoded full output string (prompt + generated text).
    We later strip the prompt prefix to isolate the prediction.
    """
    prompt = build_inference_prompt(messages)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,          # greedy decode for deterministic eval
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (skip the prompt)
    prompt_len  = inputs["input_ids"].shape[1]
    new_tokens  = outputs[0][prompt_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_reference(messages: List[Dict]) -> str:
    """Extract the ground-truth assistant response from the last message."""
    # The last message is always the assistant turn in this dataset
    return messages[-1]["content"]


def compute_metrics(
    predictions: List[str],
    references: List[str],
) -> Dict[str, float]:
    """
    Compute ROUGE, corpus BLEU, and SacreBLEU for a batch of predictions.

    Parameters
    ----------
    predictions : list of model-generated strings
    references  : list of ground-truth strings (same order)

    Returns
    -------
    dict with keys: rouge1, rouge2, rougeL, bleu, sacrebleu
    """
    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    rouge1_scores, rouge2_scores, rougeL_scores = [], [], []
    for pred, ref in zip(predictions, references):
        scores = scorer.score(ref, pred)
        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    # NLTK corpus BLEU (tokenise at word level for simplicity)
    ref_tokenised  = [[ref.split()] for ref in references]
    pred_tokenised = [pred.split() for pred in predictions]
    bleu = corpus_bleu(
        ref_tokenised,
        pred_tokenised,
        smoothing_function=SmoothingFunction().method1,
    )

    # SacreBLEU (standardised, language-agnostic)
    sacre = sacrebleu.corpus_bleu(predictions, [references])

    return {
        "rouge1"   : round(sum(rouge1_scores) / len(rouge1_scores), 4),
        "rouge2"   : round(sum(rouge2_scores) / len(rouge2_scores), 4),
        "rougeL"   : round(sum(rougeL_scores) / len(rougeL_scores), 4),
        "bleu"     : round(bleu, 4),
        "sacrebleu": round(sacre.score, 2),   # sacrebleu returns 0-100 scale
    }


## 8. Baseline Evaluation (Pre-Training)

Run the **untrained** model on the test set to get baseline scores.  
These scores will be compared with post-training scores to measure improvement.


In [ ]:
from tqdm import tqdm

def evaluate_model(model, data: List[Dict], label: str = "Evaluation") -> Dict[str, float]:
    """
    Generate predictions for every example in `data` and compute metrics.

    Parameters
    ----------
    model : the (possibly fine-tuned) language model
    data  : list of raw examples (each has a "messages" key)
    label : display label for the progress bar

    Returns
    -------
    dict of metric scores
    """
    predictions, references = [], []

    for example in tqdm(data, desc=label):
        pred = predict(example["messages"], model)
        ref  = extract_reference(example["messages"])
        predictions.append(pred)
        references.append(ref)

    metrics = compute_metrics(predictions, references)
    return metrics


print("Running baseline evaluation on the test set...")
baseline_metrics = evaluate_model(model, test_data, label="Baseline")

print("\n── Baseline Metrics (pre-training) ──")
for k, v in baseline_metrics.items():
    print(f"  {k:<12}: {v}")


In [ ]:
import json

def save_metrics(metrics: Dict, path: str) -> None:
    """Write a metrics dict to a JSON file."""
    with open(path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Saved metrics to {path}")


baseline_path = f"{RESULTS_DIR}/baseline_metrics.json"
save_metrics(baseline_metrics, baseline_path)


## 9. LoRA Setup & Training

We use **LoRA (Low-Rank Adaptation)** — injecting small trainable matrices into the attention layers while keeping the base model frozen.  
This drastically reduces VRAM usage and training time.

### Why no `bitsandbytes`?
`bitsandbytes` relies on CUDA-specific kernels and **does not work on ROCm**.  
We use full bf16 weights instead, which is efficient on AMD MI-series GPUs.


In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

def setup_lora(model, lora_r: int, lora_alpha: int,
               lora_dropout: float, target_modules: List[str]):
    """
    Wrap the model with LoRA adapters using the given config.

    LoRA freezes base weights and adds low-rank trainable matrices A and B
    to the specified linear layers. Only these matrices are updated during training.

    Parameters
    ----------
    model          : base causal LM
    lora_r         : rank of the adapter (smaller = fewer params)
    lora_alpha     : scaling factor (often 2 * r)
    lora_dropout   : dropout applied inside LoRA layers
    target_modules : list of layer names to adapt

    Returns
    -------
    peft_model with LoRA adapters attached
    """
    lora_config = LoraConfig(
        r              = lora_r,
        lora_alpha     = lora_alpha,
        lora_dropout   = lora_dropout,
        bias           = "none",       # don't train biases
        task_type      = "CAUSAL_LM",
        target_modules = target_modules,
    )
    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    return peft_model


model = setup_lora(
    model,
    lora_r          = LORA_R,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    target_modules  = LORA_TARGET_MODULES,
)


In [ ]:
def build_training_args(output_dir: str) -> TrainingArguments:
    """
    Build HuggingFace TrainingArguments.

    ROCm note:
    - fp16=False  : fp16 can be unstable on some AMD GPUs; bf16 is preferred.
    - bf16=True   : native precision for AMD MI-series (CDNA architecture).
    - optim       : 'adamw_torch' is used instead of 'paged_adamw_8bit'
                    because paged optimisers rely on bitsandbytes (CUDA-only).
    """
    return TrainingArguments(
        output_dir                  = output_dir,
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUM_STEPS,
        learning_rate               = LEARNING_RATE,
        bf16                        = True,          # ROCm-native precision
        fp16                        = False,         # disable fp16 on ROCm
        logging_steps               = LOGGING_STEPS,
        eval_steps                  = EVAL_STEPS,
        save_steps                  = SAVE_STEPS,
        eval_strategy               = "steps",
        save_strategy               = "steps",
        load_best_model_at_end      = True,          # keep the best checkpoint
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        report_to                   = "none",        # disable wandb / tensorboard
        dataloader_num_workers      = 0,             # safe default on ROCm
        remove_unused_columns       = False,         # keep all dataset columns
    )


training_args = build_training_args(CHECKPOINT_DIR)
print("Training arguments ready.")


In [ ]:
from transformers import DataCollatorForLanguageModeling

# ── Data collator ─────────────────────────────────────────────────────────────
# DataCollatorForLanguageModeling is the correct collator for causal LM fine-
# tuning.  It:
#   • Pads each batch to the longest sequence in that batch (dynamic padding).
#   • Sets labels = input_ids automatically (standard CLM training objective).
#   • mlm=False tells it we are doing causal LM, not masked LM.
#
# WHY NOT DataCollatorForSeq2Seq?
#   DataCollatorForSeq2Seq expects both encoder inputs and decoder labels
#   (seq2seq models).  For a decoder-only model like Qwen it causes shape
#   mismatches and the "excessive nesting" ValueError.
data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,      # causal LM (not masked LM)
    pad_to_multiple_of = 8, # aligns sequence lengths for faster ROCm matmuls
)

# ── Trainer ───────────────────────────────────────────────────────────────────
# Note: `tokenizer` was removed from Trainer.__init__ in transformers >= 4.46.
# Padding / tokenization is handled entirely by the DataCollator above.
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    data_collator = data_collator,
)

print("Trainer initialised. Starting training...")
trainer.train()
print("Training complete.")


## 10. Post-Training Evaluation

Re-run the same test set with the fine-tuned model and compare against baseline.


In [ ]:
print("Running post-training evaluation on the test set...")
final_metrics = evaluate_model(model, test_data, label="Post-Training")

print("\n── Final Metrics (post-training) ──")
for k, v in final_metrics.items():
    print(f"  {k:<12}: {v}")


In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
print("\n── Metric Improvement (baseline → fine-tuned) ──")
print(f"  {'Metric':<12}  {'Baseline':>10}  {'Fine-tuned':>12}  {'Delta':>8}")
print("  " + "-" * 46)
for k in baseline_metrics:
    b = baseline_metrics[k]
    f = final_metrics[k]
    delta = f - b
    sign  = "+" if delta >= 0 else ""
    print(f"  {k:<12}  {b:>10.4f}  {f:>12.4f}  {sign}{delta:>7.4f}")


In [ ]:
# ── Save final metrics ────────────────────────────────────────────────────────
final_path      = f"{RESULTS_DIR}/final_metrics.json"
comparison_path = f"{RESULTS_DIR}/metrics_comparison.json"

save_metrics(final_metrics, final_path)
save_metrics(
    {"baseline": baseline_metrics, "fine_tuned": final_metrics},
    comparison_path,
)


## 11. Save LoRA Adapter

We save **only the LoRA adapter weights** (not the full model), which is small and portable.  
To run inference later, load the base model and merge the adapter via `PeftModel.from_pretrained`.


In [ ]:
def save_adapter(model, tokenizer, save_path: str) -> None:
    """
    Save the LoRA adapter weights and tokenizer to disk.

    These adapter weights are ~10–50 MB (vs several GB for the full model).
    Load later with: PeftModel.from_pretrained(base_model, save_path)
    """
    print(f"Saving LoRA adapter to {save_path} ...")
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print("Adapter saved.")


save_adapter(model, tokenizer, ADAPTER_SAVE_DIR)


In [ ]:
# ── How to reload and run inference later ─────────────────────────────────────
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# import torch
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SAVE_DIR)
# model     = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_DIR)
# model.eval()
#
# Then call predict(messages, model) as normal.
print("Done! All outputs saved to:", RESULTS_DIR)
print("Adapter saved to          :", ADAPTER_SAVE_DIR)
